In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("json", data_files="./train_pair_1w.json", split="train")
dataset

Dataset({
    features: ['sentence1', 'sentence2', 'label'],
    num_rows: 10000
})

In [4]:
dataset[0]

{'sentence1': '找一部小时候的动画片', 'sentence2': '求一部小时候的动画片。谢了', 'label': '1'}

In [5]:
datasets = dataset.train_test_split(test_size=0.2)
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 2000
    })
})

In [6]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

def process_function(examples):
    tokenized_examples = tokenizer(examples["sentence1"], examples["sentence2"], max_length=128, truncation=True)
    tokenized_examples["labels"] = [float(label) for label in examples["label"]]
    return tokenized_examples

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

Map: 100%|██████████| 2000/2000 [00:00<00:00, 20049.21 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [7]:
print(tokenized_datasets['train'][0])

{'input_ids': [101, 671, 855, 4184, 2339, 3633, 1762, 4184, 2970, 671, 763, 849, 725, 3221, 794, 3717, 7027, 1139, 3341, 4638, 691, 6205, 511, 102, 4184, 2339, 6206, 1762, 3717, 704, 3952, 3807, 511, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 0.0}


In [8]:
model = AutoModelForSequenceClassification.from_pretrained("hfl/chinese-macbert-base", num_labels=1)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/chinese-macbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metirc = evaluate.load("f1")

In [10]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = [int(p.item() > 0.5) for p in predictions]
    labels = [int(l) for l in labels]
    # predictions = predictions.argmax(axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metirc.compute(predictions=predictions, references=labels)
    acc.update(f1)
    return acc

In [11]:
train_args = TrainingArguments(output_dir="./cross_model",      # 输出文件夹
                               per_device_train_batch_size=48,  # 训练时的batch_size
                               per_device_eval_batch_size=48,   # 验证时的batch_size
                               logging_steps=50,                # log 打印的频率
                               eval_strategy="steps",           # 评估策略
                               save_strategy="steps",           # 保存策略
                               save_total_limit=3,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True)     # 训练完成后加载最优模型

In [12]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model, 
                  args=train_args, 
                  tokenizer=tokenizer,
                  train_dataset=tokenized_datasets["train"], 
                  eval_dataset=tokenized_datasets["test"], 
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

C:\Users\10433\AppData\Local\Temp\ipykernel_24976\189788917.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


In [13]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1
50,0.156500,0.107316,0.851500,0.808511
100,0.123600,0.097151,0.868500,0.826632
150,0.119500,0.084346,0.884500,0.851638
200,0.094000,0.080408,0.893000,0.864041
250,0.088400,0.073184,0.900500,0.875235
300,0.081100,0.071400,0.899500,0.872056
350,0.069100,0.070691,0.909500,0.887087
400,0.059500,0.077549,0.901000,0.872587
450,0.060200,0.069657,0.908000,0.886978
500,0.061800,0.068893,0.908500,0.887246


TrainOutput(global_step=501, training_loss=0.09130221672222286, metrics={'train_runtime': 241.1483, 'train_samples_per_second': 99.524, 'train_steps_per_second': 2.078, 'total_flos': 1572682879479744.0, 'train_loss': 0.09130221672222286, 'epoch': 3.0})

In [14]:
trainer.evaluate(tokenized_datasets["test"])

{'eval_loss': 0.06889283657073975,
 'eval_accuracy': 0.9085,
 'eval_f1': 0.8872458410351202,
 'eval_runtime': 5.7741,
 'eval_samples_per_second': 346.372,
 'eval_steps_per_second': 7.274,
 'epoch': 3.0}

In [15]:
from transformers import pipeline, TextClassificationPipeline

In [16]:
model.config.id2label = {0: "不相似", 1: "相似"}

In [17]:
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [18]:
result = pipe({"text": "我喜欢北京", "text_pair": "天气怎样"}, function_to_apply="none")
result["label"] = "相似" if result["score"] > 0.5 else "不相似"
result

{'label': '不相似', 'score': -0.009737669490277767}